# Membro 2 — ResNet / ResNeXt
## Aplicação de Sistemas Inteligentes — Comparativo de Arquiteturas CNN

| Variação | Peso | Backbone | Épocas | Resolução | LR |
|---|---|---|---|---|---|
| V1 | Muito Leve | ResNet-50 100% congelado | 12 | 224 | 1e-3 |
| V2 | Muito Leve | ResNeXt-50 100% congelado | 12 | 224 | 1e-3 |
| V3 | Leve | ResNet-50 layer4 livre | 22 | 224 | 1e-4 |
| V4 | Leve | ResNet-101 layer4 livre + Scheduler | 25 | 224 | 1e-4 |
| V5 | Médio | ResNet-50 layer3+4 livres | 35 | 224 | 5e-5 |
| V6 | Médio | ResNeXt-50 layer3+4 + 256px | 38 | 256 | 5e-5 |
| V7 | Pesado | ResNet-50 FT completo + TTA | 50 | 256 | 1e-5 |
| V8 | Pesado | ResNeXt-50 FT completo + TTA | 60 | 256 | 1e-5 |

> **Registrar para cada variação:** Accuracy · F1 Macro · Precision · Recall · Loss · Tempo (min) · curvas treino/val · matriz de confusão


## 1 · Instalação de dependências

In [ ]:
import importlib

required = [
    "torch",
    "torchvision",
    "timm",
    "sklearn",
    "matplotlib",
]

missing = []

for pkg in required:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

if missing:
    print("Pacotes faltando:", missing)
else:
    print("✅ Todas as dependências instaladas")


## 2 · Imports e configuração global

In [ ]:
import os, time, itertools, joblib
from pathlib import Path
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import timm
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import (
    resnet50,  ResNet50_Weights,
    resnet101, ResNet101_Weights,
    resnext50_32x4d, ResNeXt50_32X4D_Weights,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 3 · Configuração do dataset
> ⚠️ **Ajuste `DATA_ROOT` para apontar para a sua pasta de imagens.**
> A pasta deve ter subpastas por classe (padrão `ImageFolder`).

In [ ]:
# ─── CONFIGURE AQUI ───────────────────────────────────────────────────────
DATA_ROOT = Path("..\\data_sets\\ovarian_ultrasound_dataset")
# ──────────────────────────────────────────────────────────────────────────

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"\n❌ Pasta não encontrada: {DATA_ROOT.resolve()}"
        f"\n   Ajuste DATA_ROOT na célula acima."
    )

# Detecta se já há split train/validation pré-definido
use_predefined_split = (
    (DATA_ROOT / "train").exists() and
    (DATA_ROOT / "validation").exists()
)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Verificação rápida de classes
_probe = datasets.ImageFolder(DATA_ROOT / "train" if use_predefined_split else DATA_ROOT)
CLASS_NAMES = _probe.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"Dataset  : {DATA_ROOT.resolve()}")
print(f"Classes  : {CLASS_NAMES}  ({NUM_CLASSES} classes)")
print(f"Split    : {'pré-definido (train/validation)' if use_predefined_split else 'gerado automaticamente 80/20 estratificado'}")


## 4 · Pipeline de dados (modular)
As transformações são definidas por nível de augmentation para reaproveitamento em todas as variações.

In [ ]:
def get_transforms(aug_level: str, img_size: int = 224):
    """
    aug_level:
        'minimal'  → flip H
        'minimal_rot' → flip H + rotação ±10°  (V2 EfficientNet)
        'light'    → flip H/V + rot ±15°
        'light_jitter' → flip + jitter cor leve
        'moderate' → flip + rot + jitter cor + blur
        'moderate_crop' → flip + rot + jitter + RandomCrop
        'heavy'    → flip + rot + jitter + RandomErasing
    """
    normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    aug_map = {
        "minimal": [
            transforms.RandomHorizontalFlip(),
        ],
        "minimal_rot": [
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
        ],
        "light": [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
        ],
        "light_jitter": [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
        ],
        "moderate": [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        ],
        "moderate_crop": [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        ],
        "heavy": [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        ],
    }

    pre_tensor = [transforms.Resize((img_size, img_size))] + aug_map[aug_level]
    post_tensor = [transforms.ToTensor(), normalize]

    if aug_level == "heavy":
        post_tensor.append(transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)))

    train_tfms = transforms.Compose(pre_tensor + post_tensor)
    val_tfms   = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_tfms, val_tfms


def make_loaders(train_tfms, val_tfms, batch_size=32):
    """Cria DataLoaders respeitando split pré-definido ou estratificado."""
    if use_predefined_split:
        train_ds = datasets.ImageFolder(DATA_ROOT / "train",      transform=train_tfms)
        val_ds   = datasets.ImageFolder(DATA_ROOT / "validation", transform=val_tfms)
    else:
        full_ds = datasets.ImageFolder(DATA_ROOT)
        targets = [y for _, y in full_ds.samples]
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, val_idx = next(sss.split(np.zeros(len(targets)), targets))

        train_ds_src = datasets.ImageFolder(DATA_ROOT, transform=train_tfms)
        val_ds_src   = datasets.ImageFolder(DATA_ROOT, transform=val_tfms)
        train_ds     = Subset(train_ds_src, train_idx)
        val_ds       = Subset(val_ds_src,   val_idx)

    nw = min(4, os.cpu_count() or 1) if device.type == "cuda" else 0
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=nw, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=nw, pin_memory=True)
    return train_loader, val_loader

print("✅ Pipeline de dados pronto")


## 5 · Utilitários de treinamento

In [ ]:
# ─── Loop de época ────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer=None, criterion=None, train=True, scheduler=None):
    model.train(mode=train)
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []

    with torch.set_grad_enabled(train):
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()
            preds = logits.argmax(1)
            total_loss     += loss.item() * y.size(0)
            total_correct  += (preds == y).sum().item()
            total          += y.size(0)
            all_preds.append(preds.detach().cpu().numpy())
            all_targets.append(y.detach().cpu().numpy())

    avg_loss = total_loss / total
    acc      = total_correct / total
    y_true   = np.concatenate(all_targets)
    y_pred   = np.concatenate(all_preds)
    f1       = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return avg_loss, acc, f1, y_true, y_pred


# ─── TTA (Test-Time Augmentation) ─────────────────────────────────────────
def tta_predict(model, loader, img_size=256, n_aug=4):
    """Flip H + crop central + rotação ±5° — média das probabilidades."""
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            probs = torch.softmax(model(X), dim=1)
            # flip horizontal
            probs_flip = torch.softmax(model(torch.flip(X, dims=[3])), dim=1)
            probs_avg = (probs + probs_flip) / 2
            all_probs.append(probs_avg.cpu())
            all_targets.append(y.cpu())
    probs_stack = torch.cat(all_probs, dim=0)
    targets_all  = torch.cat(all_targets, dim=0).numpy()
    preds = probs_stack.argmax(1).numpy()
    f1 = f1_score(targets_all, preds, average="macro", zero_division=0)
    acc = (preds == targets_all).mean()
    return acc, f1, targets_all, preds


# ─── Treinamento completo ──────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, epochs, lr,
                scheduler_fn=None, label_smoothing=0.0, weight_decay=0.0,
                var_name="V?", use_tta=False, img_size=224, patience=25):
    """
    Treina e valida o modelo, registra histórico e retorna resultados.
    scheduler_fn: callable(optimizer) → scheduler  |  None
    Usa early stopping baseado em val_loss (patience).
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay
    )
    scheduler = scheduler_fn(optimizer) if scheduler_fn else None

    history = {k: [] for k in ["train_loss","val_loss","train_acc","val_acc","val_f1"]}
    best_state = None
    best_loss  = float('inf')
    best_f1    = 0.0
    best_yt, best_yp = None, None
    t0 = time.time()
    epochs_no_improve = 0

    for e in range(1, epochs + 1):
        tr_loss, tr_acc, _, _, _         = run_epoch(model, train_loader, optimizer, criterion, train=True)
        va_loss, va_acc, va_f1, yt, yp   = run_epoch(model, val_loader,   optimizer, criterion, train=False)

        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(va_loss)
            else:
                scheduler.step()

        for k, v in zip(history.keys(), [tr_loss, va_loss, tr_acc, va_acc, va_f1]):
            history[k].append(v)

        if va_loss < best_loss:
            best_loss  = va_loss
            best_f1    = va_f1
            best_state = deepcopy(model.state_dict())
            best_yt, best_yp = yt, yp
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stop na época {e}")
                break

        print(f"  [{e:>3}/{epochs}] loss {tr_loss:.4f}/{va_loss:.4f}  "
              f"acc {tr_acc:.4f}/{va_acc:.4f}  f1_val {va_f1:.4f}")

    elapsed = (time.time() - t0) / 60
    model.load_state_dict(best_state)

    if use_tta:
        print("  → Aplicando TTA...")
        tta_acc, tta_f1, best_yt, best_yp = tta_predict(model, val_loader, img_size)
        print(f"  TTA  acc={tta_acc:.4f}  f1={tta_f1:.4f}")
        final_acc, final_f1 = tta_acc, tta_f1
    else:
        final_acc = history["val_acc"][-1]
        final_f1  = best_f1

    precision = precision_score(best_yt, best_yp, average="macro", zero_division=0)
    recall    = recall_score   (best_yt, best_yp, average="macro", zero_division=0)

    results = {
        "variacao": var_name,
        "accuracy": round(final_acc, 4),
        "f1_macro": round(final_f1,  4),
        "precision": round(precision, 4),
        "recall":   round(recall,    4),
        "loss":     round(best_loss, 4),
        "tempo_min": round(elapsed, 1),
    }
    return model, history, results, best_yt, best_yp


# ─── Plots ────────────────────────────────────────────────────────────────
def plot_curves(history, var_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history["train_loss"], label="train"); ax1.plot(history["val_loss"], label="val")
    ax1.set_title(f"{var_name} — Loss"); ax1.set_xlabel("epoch"); ax1.legend()
    ax2.plot(history["train_acc"],  label="train"); ax2.plot(history["val_acc"],  label="val")
    ax2.set_title(f"{var_name} — Accuracy"); ax2.set_xlabel("epoch"); ax2.legend()
    plt.tight_layout(); plt.show()

def plot_cm(y_true, y_pred, class_names, var_name, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(max(5, len(class_names)*1.2), max(4, len(class_names))))
    plt.imshow(cm, cmap="Blues")
    plt.title(f"{var_name} — Matriz de Confusão {'Normalizada' if normalize else ''}")
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right")
    plt.yticks(ticks, class_names)
    fmt = ".2f" if normalize else "d"
    thresh = cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i,j], fmt), ha="center",
                 color="white" if cm[i,j] > thresh else "black", fontsize=9)
    plt.ylabel("Real"); plt.xlabel("Previsto")
    plt.tight_layout(); plt.show()

def show_results(res):
    print(f"\n{'─'*55}")
    print(f"  {res['variacao']} — Resultados Finais")
    print(f"{'─'*55}")
    print(f"  Accuracy   : {res['accuracy']:.4f}")
    print(f"  F1 Macro   : {res['f1_macro']:.4f}")
    print(f"  Precision  : {res['precision']:.4f}")
    print(f"  Recall     : {res['recall']:.4f}")
    print(f"  Loss (val) : {res['loss']:.4f}")
    print(f"  Tempo      : {res['tempo_min']} min")
    print(f"{'─'*55}")

print("✅ Utilitários de treinamento prontos")


## 6 · Tabela de resultados
Acumula todas as variações automaticamente.

In [ ]:
ALL_RESULTS = []   # preenchido automaticamente após cada variação

def print_results_table():
    if not ALL_RESULTS:
        print("Nenhum resultado ainda.")
        return
    header = f"{'Var':<6} {'Backbone':<30} {'Acc':>6} {'F1':>6} {'Prec':>6} {'Rec':>6} {'Loss':>6} {'Tempo':>6}"
    print(header)
    print("─" * len(header))
    for r in ALL_RESULTS:
        print(f"{r['variacao']:<6} {r.get('backbone','—'):<30} "
              f"{r['accuracy']:>6.4f} {r['f1_macro']:>6.4f} "
              f"{r['precision']:>6.4f} {r['recall']:>6.4f} "
              f"{r['loss']:>6.4f} {r['tempo_min']:>5.1f}m")

print("✅ Tabela pronta — execute print_results_table() a qualquer momento")


In [ ]:
# ─── Salvamento de modelos ────────────────────────────────────────────────
SAVE_DIR = Path("modelos_salvos") / "membro2"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def save_model(model, var_name: str) -> Path:
    """
    Salva o modelo completo em .joblib dentro de SAVE_DIR.
    Nome do arquivo: membro2_<var_name_sanitizado>.joblib
    """
    safe = var_name.replace(" ", "_").replace("/", "-")
    path = SAVE_DIR / f"membro2_{safe}.joblib"
    joblib.dump(model, path)
    print(f"  💾 Modelo salvo → {path}")
    return path

print(f"✅ Função save_model pronta  |  pasta: {SAVE_DIR.resolve()}")


---
## V1 · Muito Leve — ResNet-50 (100% congelado)

| Parâmetro | Valor |
|---|---|
| Backbone | ResNet-50, ImageNet, 100% congelado |
| Cabeça | AvgPool + FC (linear) |
| Augmentation | Flip H |
| Scheduler | Nenhum |
| Épocas | 12 |
| Resolução | 224 |
| LR | 1e-3 |

**Objetivo:** baseline ResNet clássico, ponto-zero de referência.


In [ ]:
def build_v1():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights)
    for p in model.parameters():
        p.requires_grad = False
    in_feats = model.fc.in_features
    model.fc = nn.Linear(in_feats, NUM_CLASSES)
    return model.to(device)

train_tfms_v1, val_tfms_v1 = get_transforms("minimal", img_size=224)
train_loader_v1, val_loader_v1 = make_loaders(train_tfms_v1, val_tfms_v1, batch_size=32)

model_v1 = build_v1()
trainable_v1 = sum(p.numel() for p in model_v1.parameters() if p.requires_grad)
total_v1     = sum(p.numel() for p in model_v1.parameters())
print(f"Parâmetros treináveis: {trainable_v1:,} / {total_v1:,}")


In [ ]:
print("\n=== Treinando V1 ===")
model_v1, hist_v1, res_v1, yt_v1, yp_v1 = train_model(
    model_v1, train_loader_v1, val_loader_v1,
    epochs=1000,
    lr=1e-3,
    var_name="V1 ResNet-50 frozen",
)
res_v1["backbone"] = "ResNet-50 (frozen)"
ALL_RESULTS.append(res_v1)
show_results(res_v1)

save_model(model_v1, res_v1["variacao"])

In [ ]:
plot_curves(hist_v1, "V1 — ResNet-50 (frozen)")
plot_cm(yt_v1, yp_v1, CLASS_NAMES, "V1", normalize=True)
print(classification_report(yt_v1, yp_v1, target_names=CLASS_NAMES))


> **Observação V1:** *(preencher após execução)*

---
## V2 · Muito Leve — ResNeXt-50 (100% congelado)

| Parâmetro | Valor |
|---|---|
| Backbone | ResNeXt-50_32x4d, ImageNet, 100% congelado |
| Cabeça | AvgPool + FC |
| Augmentation | Flip H |
| Scheduler | Nenhum |
| Épocas | 12 |
| Resolução | 224 |
| LR | 1e-3 |

**Objetivo:** comparar ResNet-50 vs ResNeXt-50 com mesma configuração.


In [ ]:
def build_v2():
    weights = ResNeXt50_32X4D_Weights.IMAGENET1K_V2
    model = resnext50_32x4d(weights=weights)
    for p in model.parameters():
        p.requires_grad = False
    in_feats = model.fc.in_features
    model.fc = nn.Linear(in_feats, NUM_CLASSES)
    return model.to(device)

train_tfms_v2, val_tfms_v2 = get_transforms("minimal", img_size=224)
train_loader_v2, val_loader_v2 = make_loaders(train_tfms_v2, val_tfms_v2, batch_size=32)

model_v2 = build_v2()
print(f"Parâmetros treináveis: {sum(p.numel() for p in model_v2.parameters() if p.requires_grad):,}")


In [ ]:
print("\n=== Treinando V2 ===")
model_v2, hist_v2, res_v2, yt_v2, yp_v2 = train_model(
    model_v2, train_loader_v2, val_loader_v2,
    epochs=1000,
    lr=1e-3,
    var_name="V2 ResNeXt-50 frozen",
)
res_v2["backbone"] = "ResNeXt-50 (frozen)"
ALL_RESULTS.append(res_v2)
show_results(res_v2)

save_model(model_v2, res_v2["variacao"])

In [ ]:
plot_curves(hist_v2, "V2 — ResNeXt-50 (frozen)")
plot_cm(yt_v2, yp_v2, CLASS_NAMES, "V2", normalize=True)
print(classification_report(yt_v2, yp_v2, target_names=CLASS_NAMES))


> **Observação V2:** *(preencher após execução)*

---
## V3 · Leve — ResNet-50 (layer4 livre)

| Parâmetro | Valor |
|---|---|
| Backbone | ResNet-50 — layer4 livre |
| Cabeça | FC(512) + Dropout(0.3) + FC |
| Augmentation | Flip H/V + Rotação ±15° |
| Scheduler | Nenhum |
| Épocas | 22 |
| Resolução | 224 |
| LR | 1e-4 |

**Objetivo:** primeiro fine-tuning sutil — layer4 (último grupo residual) descongelado.


In [ ]:
def build_v3():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layer4.parameters():
        p.requires_grad = True
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_feats, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, NUM_CLASSES),
    )
    return model.to(device)

train_tfms_v3, val_tfms_v3 = get_transforms("light", img_size=224)
train_loader_v3, val_loader_v3 = make_loaders(train_tfms_v3, val_tfms_v3, batch_size=32)

model_v3 = build_v3()
print(f"Parâmetros treináveis: {sum(p.numel() for p in model_v3.parameters() if p.requires_grad):,}")


In [ ]:
print("\n=== Treinando V3 ===")
model_v3, hist_v3, res_v3, yt_v3, yp_v3 = train_model(
    model_v3, train_loader_v3, val_loader_v3,
    epochs=1000,
    lr=1e-4,
    var_name="V3 ResNet-50 layer4",
)
res_v3["backbone"] = "ResNet-50 (layer4)"
ALL_RESULTS.append(res_v3)
show_results(res_v3)

save_model(model_v3, res_v3["variacao"])

In [ ]:
plot_curves(hist_v3, "V3 — ResNet-50 (layer4 livre)")
plot_cm(yt_v3, yp_v3, CLASS_NAMES, "V3", normalize=True)
print(classification_report(yt_v3, yp_v3, target_names=CLASS_NAMES))


> **Observação V3:** *(preencher após execução)*

---
## V4 · Leve — ResNet-101 (layer4 + Scheduler)

| Parâmetro | Valor |
|---|---|
| Backbone | **ResNet-101** — layer4 livre |
| Cabeça | FC(512) + Dropout(0.35) + FC |
| Augmentation | Flip + Rot + ColorJitter leve |
| Scheduler | ReduceLROnPlateau (patience=4) |
| Épocas | 25 |
| Resolução | 224 |
| LR | 1e-4 |

**Objetivo:** avaliar ganho de profundidade (50 → 101) com scheduler.


In [ ]:
def build_v4():
    weights = ResNet101_Weights.IMAGENET1K_V2
    model = resnet101(weights=weights)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layer4.parameters():
        p.requires_grad = True
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_feats, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.35),
        nn.Linear(512, NUM_CLASSES),
    )
    return model.to(device)

train_tfms_v4, val_tfms_v4 = get_transforms("light_jitter", img_size=224)
train_loader_v4, val_loader_v4 = make_loaders(train_tfms_v4, val_tfms_v4, batch_size=24)

model_v4 = build_v4()
print(f"Parâmetros treináveis: {sum(p.numel() for p in model_v4.parameters() if p.requires_grad):,}")


In [ ]:
print("\n=== Treinando V4 ===")
model_v4, hist_v4, res_v4, yt_v4, yp_v4 = train_model(
    model_v4, train_loader_v4, val_loader_v4,
    epochs=1000,
    lr=1e-4,
    scheduler_fn=lambda opt: torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=4, factor=0.5),
    var_name="V4 ResNet-101 +Scheduler",
)
res_v4["backbone"] = "ResNet-101 (layer4+Sched)"
ALL_RESULTS.append(res_v4)
show_results(res_v4)

save_model(model_v4, res_v4["variacao"])

In [ ]:
plot_curves(hist_v4, "V4 — ResNet-101 (Scheduler)")
plot_cm(yt_v4, yp_v4, CLASS_NAMES, "V4", normalize=True)
print(classification_report(yt_v4, yp_v4, target_names=CLASS_NAMES))


> **Observação V4:** *(preencher após execução)*

---
## V5 · Médio — ResNet-50 (layer3+4 livres)

| Parâmetro | Valor |
|---|---|
| Backbone | ResNet-50 — layer3 + layer4 livres |
| Cabeça | FC(1024) + BN + Dropout(0.4) + FC |
| Augmentation | Flip + Rot + Jitter + Blur |
| Scheduler | ReduceLROnPlateau (patience=5) |
| Épocas | 35 |
| Resolução | 224 |
| LR | 5e-5 |

**Objetivo:** descongelar 2 blocos — monitorar risco de overfitting.


In [ ]:
def build_v5():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layer3.parameters():
        p.requires_grad = True
    for p in model.layer4.parameters():
        p.requires_grad = True
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_feats, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(1024, NUM_CLASSES),
    )
    return model.to(device)

train_tfms_v5, val_tfms_v5 = get_transforms("moderate", img_size=224)
train_loader_v5, val_loader_v5 = make_loaders(train_tfms_v5, val_tfms_v5, batch_size=24)

model_v5 = build_v5()
print(f"Parâmetros treináveis: {sum(p.numel() for p in model_v5.parameters() if p.requires_grad):,}")


In [ ]:
print("\n=== Treinando V5 ===")
model_v5, hist_v5, res_v5, yt_v5, yp_v5 = train_model(
    model_v5, train_loader_v5, val_loader_v5,
    epochs=1000,
    lr=5e-5,
    scheduler_fn=lambda opt: torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=5, factor=0.5),
    var_name="V5 ResNet-50 layer3+4",
)
res_v5["backbone"] = "ResNet-50 (layer3+4)"
ALL_RESULTS.append(res_v5)
show_results(res_v5)

save_model(model_v5, res_v5["variacao"])

In [ ]:
plot_curves(hist_v5, "V5 — ResNet-50 (layer3+4)")
plot_cm(yt_v5, yp_v5, CLASS_NAMES, "V5", normalize=True)
print(classification_report(yt_v5, yp_v5, target_names=CLASS_NAMES))


> **Observação V5:** *(preencher após execução)*

---
## V6 · Médio — ResNeXt-50 (layer3+4 livres, 256px)

| Parâmetro | Valor |
|---|---|
| Backbone | **ResNeXt-50** — layer3 + layer4 livres |
| Cabeça | FC(1024) + BN + Dropout(0.4) + FC |
| Augmentation | Flip + Rot + Jitter + RandomCrop |
| Scheduler | StepLR (γ=0.5, step=10) |
| Épocas | 38 |
| Resolução | **256** |
| LR | 5e-5 |

**Objetivo:** ResNeXt com aug moderada forte; comparar com V5 (ResNet-50).


In [ ]:
def build_v6():
    weights = ResNeXt50_32X4D_Weights.IMAGENET1K_V2
    model = resnext50_32x4d(weights=weights)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layer3.parameters():
        p.requires_grad = True
    for p in model.layer4.parameters():
        p.requires_grad = True
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_feats, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(1024, NUM_CLASSES),
    )
    return model.to(device)

train_tfms_v6, val_tfms_v6 = get_transforms("moderate_crop", img_size=256)
train_loader_v6, val_loader_v6 = make_loaders(train_tfms_v6, val_tfms_v6, batch_size=16)

model_v6 = build_v6()
print(f"Parâmetros treináveis: {sum(p.numel() for p in model_v6.parameters() if p.requires_grad):,}")


In [ ]:
print("\n=== Treinando V6 ===")
model_v6, hist_v6, res_v6, yt_v6, yp_v6 = train_model(
    model_v6, train_loader_v6, val_loader_v6,
    epochs=1000,
    lr=5e-5,
    scheduler_fn=lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5),
    var_name="V6 ResNeXt-50 256px",
)
res_v6["backbone"] = "ResNeXt-50 (l3+4, 256)"
ALL_RESULTS.append(res_v6)
show_results(res_v6)

save_model(model_v6, res_v6["variacao"])

In [ ]:
plot_curves(hist_v6, "V6 — ResNeXt-50 (256px)")
plot_cm(yt_v6, yp_v6, CLASS_NAMES, "V6", normalize=True)
print(classification_report(yt_v6, yp_v6, target_names=CLASS_NAMES))


> **Observação V6:** *(preencher após execução)*

---
## V7/V8 · Pesado — Helpers CutMix

In [ ]:
import torch.nn.functional as F

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

def cutmix_data(x, y, alpha=0.4):
    """CutMix simples: retorna x_mix, y_a, y_b, lam."""
    lam = np.random.beta(alpha, alpha)
    rand_idx = torch.randperm(x.size(0)).to(device)
    y_a, y_b = y, y[rand_idx]
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x_mix = x.clone()
    x_mix[:, :, bbx1:bbx2, bby1:bby2] = x[rand_idx, :, bbx1:bbx2, bby1:bby2]
    lam = 1 - (bbx2-bbx1)*(bby2-bby1) / (x.size(-1)*x.size(-2))
    return x_mix, y_a, y_b, lam

def train_cutmix_epoch(model, loader, optimizer, criterion, cutmix_prob=0.5):
    """Época com CutMix aplicado com probabilidade cutmix_prob."""
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        if np.random.rand() < cutmix_prob:
            X_mix, y_a, y_b, lam = cutmix_data(X, y)
            logits = model(X_mix)
            loss = lam * criterion(logits, y_a) + (1-lam) * criterion(logits, y_b)
        else:
            logits = model(X)
            loss = criterion(logits, y)
        loss.backward(); optimizer.step()
        preds = logits.argmax(1)
        total_loss    += loss.item() * y.size(0)
        total_correct += (preds == y).sum().item()
        total         += y.size(0)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())
    avg_loss = total_loss / total
    acc      = total_correct / total
    y_true   = np.concatenate(all_targets)
    y_pred   = np.concatenate(all_preds)
    f1       = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return avg_loss, acc, f1, y_true, y_pred

print("✅ Funções CutMix prontas")


---
## V7 · Pesado — ResNet-50 (FT completo + TTA)

| Parâmetro | Valor |
|---|---|
| Backbone | ResNet-50 — **FT completo** (LR diff: backbone 1e-5 / cabeça 1e-4) |
| Cabeça | FC(2048) + BN + Dropout(0.5) + FC |
| Augmentation | CutMix + RandomErasing + Jitter |
| Scheduler | CosineAnnealingLR |
| Épocas | 50 |
| Resolução | 256 |
| Extras | WD=5e-4, Label Smoothing 0.1, TTA |

**Objetivo:** FT completo do ResNet-50 com regularização forte.


In [ ]:
def build_v7():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights)
    for p in model.parameters():
        p.requires_grad = True
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_feats, 2048),
        nn.BatchNorm1d(2048),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(2048, NUM_CLASSES),
    )
    return model.to(device)

train_tfms_v7, val_tfms_v7 = get_transforms("heavy", img_size=256)
train_loader_v7, val_loader_v7 = make_loaders(train_tfms_v7, val_tfms_v7, batch_size=16)

model_v7 = build_v7()
print(f"Parâmetros treináveis: {sum(p.numel() for p in model_v7.parameters() if p.requires_grad):,}")


In [ ]:
EPOCHS_V7 = 50
criterion_v7 = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_v7 = torch.optim.AdamW([
    {"params": list(model_v7.conv1.parameters()) + list(model_v7.bn1.parameters()) + list(model_v7.layer1.parameters()) + list(model_v7.layer2.parameters()) + list(model_v7.layer3.parameters()) + list(model_v7.layer4.parameters()), "lr": 1e-05},
    {"params": model_v7.fc.parameters(),     "lr": 0.0001},
], weight_decay=0.0005)
scheduler_v7 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_v7, T_max=EPOCHS_V7, eta_min=1e-7)

print("\n=== Treinando V7 (FT completo) ===")
history_v7 = {k: [] for k in ["train_loss","val_loss","train_acc","val_acc","val_f1"]}
best_f1_v7, best_state_v7 = 0.0, None
yt_v7, yp_v7 = None, None
t0_v7 = time.time()

for e in range(1, EPOCHS_V7 + 1):
    tr_loss, tr_acc, _, _, _       = train_cutmix_epoch(model_v7, train_loader_v7, optimizer_v7, criterion_v7)
    va_loss, va_acc, va_f1, yt, yp = run_epoch(model_v7, val_loader_v7, optimizer_v7, criterion_v7, train=False)
    scheduler_v7.step()

    for k, v in zip(history_v7.keys(), [tr_loss, va_loss, tr_acc, va_acc, va_f1]):
        history_v7[k].append(v)
    if va_f1 > best_f1_v7:
        best_f1_v7, best_state_v7 = va_f1, deepcopy(model_v7.state_dict())
        yt_v7, yp_v7 = yt, yp
    print(f"  [{e:>3}/{EPOCHS_V7}] loss {tr_loss:.4f}/{va_loss:.4f}  acc {tr_acc:.4f}/{va_acc:.4f}  f1 {va_f1:.4f}")

model_v7.load_state_dict(best_state_v7)
elapsed_v7 = (time.time() - t0_v7) / 60

print("  → Aplicando TTA...")
tta_acc_v7, tta_f1_v7, yt_v7, yp_v7 = tta_predict(model_v7, val_loader_v7, img_size=256)
print(f"  TTA  acc={tta_acc_v7:.4f}  f1={tta_f1_v7:.4f}")

res_v7 = {
    "variacao": "V7 ResNet-50 FT+TTA",
    "backbone": "ResNet-50 (FT completo, TTA)",
    "accuracy": round(tta_acc_v7, 4),
    "f1_macro": round(tta_f1_v7,  4),
    "precision": round(precision_score(yt_v7, yp_v7, average="macro", zero_division=0), 4),
    "recall":    round(recall_score(yt_v7, yp_v7, average="macro", zero_division=0), 4),
    "loss":      round(history_v7["val_loss"][-1], 4),
    "tempo_min": round(elapsed_v7, 1),
}
ALL_RESULTS.append(res_v7)
show_results(res_v7)

save_model(model_v7, res_v7["variacao"])


In [ ]:
plot_curves(history_v7, "V7 — ResNet-50 (FT completo)")
plot_cm(yt_v7, yp_v7, CLASS_NAMES, "V7", normalize=True)
print(classification_report(yt_v7, yp_v7, target_names=CLASS_NAMES))


> **Observação V7:** *(preencher após execução)*

---
## V8 · Pesado — ResNeXt-50 (FT completo + RandAugment + Label Smoothing)

| Parâmetro | Valor |
|---|---|
| Backbone | **ResNeXt-50** — FT completo (LR diff) |
| Cabeça | FC(2048) + BN + Dropout(0.5) + FC |
| Augmentation | CutMix + RandAugment |
| Scheduler | CosineAnnealingWarmRestarts |
| Épocas | 60 |
| Resolução | 256 |
| Extras | Label Smoothing 0.1, WD=1e-4, TTA |

**Objetivo:** principal candidato ao ensemble — ResNeXt + regularização máxima.


In [ ]:
def build_v8():
    weights = ResNeXt50_32X4D_Weights.IMAGENET1K_V2
    model = resnext50_32x4d(weights=weights)
    for p in model.parameters():
        p.requires_grad = True
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_feats, 2048),
        nn.BatchNorm1d(2048),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(2048, NUM_CLASSES),
    )
    return model.to(device)

def get_transforms_randaugment(img_size=256):
    normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    train_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
        transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
    ])
    val_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_tfms, val_tfms

train_tfms_v8, val_tfms_v8 = get_transforms_randaugment(img_size=256)
train_loader_v8, val_loader_v8 = make_loaders(train_tfms_v8, val_tfms_v8, batch_size=16)

model_v8 = build_v8()
print(f"Parâmetros treináveis: {sum(p.numel() for p in model_v8.parameters() if p.requires_grad):,}")


In [ ]:
EPOCHS_V8 = 60
criterion_v8 = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_v8 = torch.optim.AdamW([
    {"params": list(model_v8.conv1.parameters()) + list(model_v8.bn1.parameters()) + list(model_v8.layer1.parameters()) + list(model_v8.layer2.parameters()) + list(model_v8.layer3.parameters()) + list(model_v8.layer4.parameters()), "lr": 1e-05},
    {"params": model_v8.fc.parameters(),     "lr": 0.0001},
], weight_decay=0.0001)
scheduler_v8 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer_v8, T_0=20, T_mult=2)

print("\n=== Treinando V8 (FT completo) ===")
history_v8 = {k: [] for k in ["train_loss","val_loss","train_acc","val_acc","val_f1"]}
best_f1_v8, best_state_v8 = 0.0, None
yt_v8, yp_v8 = None, None
t0_v8 = time.time()

for e in range(1, EPOCHS_V8 + 1):
    tr_loss, tr_acc, _, _, _       = train_cutmix_epoch(model_v8, train_loader_v8, optimizer_v8, criterion_v8)
    va_loss, va_acc, va_f1, yt, yp = run_epoch(model_v8, val_loader_v8, optimizer_v8, criterion_v8, train=False)
    scheduler_v8.step()

    for k, v in zip(history_v8.keys(), [tr_loss, va_loss, tr_acc, va_acc, va_f1]):
        history_v8[k].append(v)
    if va_f1 > best_f1_v8:
        best_f1_v8, best_state_v8 = va_f1, deepcopy(model_v8.state_dict())
        yt_v8, yp_v8 = yt, yp
    print(f"  [{e:>3}/{EPOCHS_V8}] loss {tr_loss:.4f}/{va_loss:.4f}  acc {tr_acc:.4f}/{va_acc:.4f}  f1 {va_f1:.4f}")

model_v8.load_state_dict(best_state_v8)
elapsed_v8 = (time.time() - t0_v8) / 60

print("  → Aplicando TTA...")
tta_acc_v8, tta_f1_v8, yt_v8, yp_v8 = tta_predict(model_v8, val_loader_v8, img_size=256)
print(f"  TTA  acc={tta_acc_v8:.4f}  f1={tta_f1_v8:.4f}")

res_v8 = {
    "variacao": "V8 ResNeXt-50 FT+TTA",
    "backbone": "ResNeXt-50 (FT completo, TTA)",
    "accuracy": round(tta_acc_v8, 4),
    "f1_macro": round(tta_f1_v8,  4),
    "precision": round(precision_score(yt_v8, yp_v8, average="macro", zero_division=0), 4),
    "recall":    round(recall_score(yt_v8, yp_v8, average="macro", zero_division=0), 4),
    "loss":      round(history_v8["val_loss"][-1], 4),
    "tempo_min": round(elapsed_v8, 1),
}
ALL_RESULTS.append(res_v8)
show_results(res_v8)

save_model(model_v8, res_v8["variacao"])


In [ ]:
plot_curves(history_v8, "V8 — ResNeXt-50 (FT completo)")
plot_cm(yt_v8, yp_v8, CLASS_NAMES, "V8", normalize=True)
print(classification_report(yt_v8, yp_v8, target_names=CLASS_NAMES))


> **Observação V8:** *(preencher após execução)*

---
## Consolidação Final — Todos os Resultados


In [ ]:
print("\n=== TABELA CONSOLIDADA ===")
print_results_table()


In [ ]:
# Gráfico comparativo de F1 Macro entre variações
if ALL_RESULTS:
    nomes = [r["variacao"] for r in ALL_RESULTS]
    f1s   = [r["f1_macro"] for r in ALL_RESULTS]
    accs  = [r["accuracy"] for r in ALL_RESULTS]

    x = np.arange(len(nomes))
    fig, ax = plt.subplots(figsize=(13, 5))
    bars = ax.bar(x - 0.2, f1s,  0.35, label="F1 Macro", color="#4C72B0")
    bars2= ax.bar(x + 0.2, accs, 0.35, label="Accuracy",  color="#55A868")
    ax.set_xticks(x); ax.set_xticklabels(nomes, rotation=30, ha="right")
    ax.set_ylim(0, 1); ax.set_ylabel("Score"); ax.legend()
    ax.set_title("Membro 2 — Comparativo de Variações (F1 Macro vs Accuracy)")
    for b in list(bars) + list(bars2):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                f"{b.get_height():.3f}", ha="center", fontsize=7)
    plt.tight_layout(); plt.show()


---
## Notas para o Relatório

Copie esta tabela para o documento do grupo após preencher todas as variações:

| Var. | Backbone | Accuracy | F1 Macro | Precision | Recall | Tempo | Observações |
|---|---|---|---|---|---|---|---|
| V1 | ResNet-50 frozen | | | | | | |
| V2 | ResNeXt-50 frozen | | | | | | |
| V3 | ResNet-50 layer4 livre | | | | | | |
| V4 | ResNet-101 layer4 + Scheduler | | | | | | |
| V5 | ResNet-50 layer3+4 livres | | | | | | |
| V6 | ResNeXt-50 layer3+4 256px | | | | | | |
| V7 | ResNet-50 FT completo + TTA | | | | | | |
| V8 | ResNeXt-50 FT completo + TTA | | | | | | |
